# Gradient Inspection, Modern TensorFlow And PyTorch

This notebook is the modern companion to `gradients-2020-tf22-compat.ipynb`. It covers the same content, but with APIs 
that are used today (2026):

- TensorFlow: `tf.GradientTape`
- PyTorch: `torch.autograd`

The goal is not to train a good model (or even a bad model!), but to show how to inspect gradients 
with respect to trainable weights and intermediate activations.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import torch
from torch import nn

np.random.seed(7)
tf.random.set_seed(7)
torch.manual_seed(7)

print('TensorFlow:', tf.__version__)
print('TensorFlow eager execution:', tf.executing_eagerly())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('MPS available:', hasattr(torch.backends, 'mps') and torch.backends.mps.is_available())

## Toy Data

Similar to the old notebook, here we generate synthetic 28x28 images 
so the notebook is reproducible without needing to download any datasets.

In [ ]:
def make_toy_images(n=64, image_shape=(28, 28), num_classes=10):
    rng = np.random.default_rng(7)
    x = rng.normal(loc=0.0, scale=0.2, size=(n, *image_shape, 1)).astype('float32')
    y = (np.arange(n) % num_classes).astype('int64')

    for i, label in enumerate(y):
        row = 2 + (label % 5) * 4
        col = 2 + (label // 5) * 10
        x[i, row:row + 4, col:col + 4, 0] += 1.0

    return x, y

x_np, y_np = make_toy_images()
x_batch_np = x_np[:16]
y_batch_np = y_np[:16]

x_tf = tf.convert_to_tensor(x_batch_np)
y_tf = tf.convert_to_tensor(y_batch_np, dtype=tf.int64)

x_torch = torch.tensor(np.transpose(x_batch_np, (0, 3, 1, 2)), dtype=torch.float32)
y_torch = torch.tensor(y_batch_np, dtype=torch.long)

print('TensorFlow batch:', x_tf.shape, y_tf.shape)
print('PyTorch batch:', x_torch.shape, y_torch.shape)

## TensorFlow: Weights And Activations With `GradientTape`

In modern TensorFlow, the normal pattern is: run the forward pass inside a tape, compute a scalar loss, then ask the tape for gradients.

In [ ]:
def build_tf_probe_model():
    inputs = keras.Input(shape=(28, 28, 1), name='image')
    conv1 = layers.Conv2D(8, 3, padding='same', use_bias=False, name='conv_1')(inputs)
    relu1 = layers.ReLU(name='relu_1')(conv1)
    pooled = layers.MaxPool2D(pool_size=2, strides=2, padding='same', name='pool_1')(relu1)
    flat = layers.Flatten(name='flatten')(pooled)
    dense1 = layers.Dense(16, activation='relu', name='dense_1')(flat)
    logits = layers.Dense(10, name='logits')(dense1)

    model = keras.Model(inputs, logits, name='tf_modern_model')
    probe_model = keras.Model(
        inputs,
        [logits, conv1, relu1, dense1],
        name='tf_modern_probe_model',
    )
    return model, probe_model

tf_model, tf_probe_model = build_tf_probe_model()
tf_loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

with tf.GradientTape(persistent=True) as tape:
    logits, conv1, relu1, dense1 = tf_probe_model(x_tf, training=True)
    loss = tf_loss_fn(y_tf, logits)

weight_grads = tape.gradient(loss, tf_model.trainable_weights)
activation_grads = tape.gradient(loss, [conv1, relu1, dense1])
del tape

print('Loss:', float(loss.numpy()))
print()
print('Weight gradients')
for weight, grad in zip(tf_model.trainable_weights, weight_grads):
    print(f'{weight.name:24} weight={tuple(weight.shape)!s:14} grad={tuple(grad.shape)!s}')

print()
print('Activation gradients')
for name, activation, grad in zip(['conv_1', 'relu_1', 'dense_1'], [conv1, relu1, dense1], activation_grads):
    print(f'{name:8} activation={tuple(activation.shape)!s:18} grad={tuple(grad.shape)!s}')

## TensorFlow: One Manual Training Step

This is the compact modern replacement for building a backend function around a symbolic loss tensor.

In [ ]:
optimizer = keras.optimizers.SGD(learning_rate=0.01)

with tf.GradientTape() as tape:
    logits = tf_model(x_tf, training=True)
    loss_before = tf_loss_fn(y_tf, logits)

grads = tape.gradient(loss_before, tf_model.trainable_weights)
optimizer.apply_gradients(zip(grads, tf_model.trainable_weights))

loss_after = tf_loss_fn(y_tf, tf_model(x_tf, training=False))
print('Loss before:', float(loss_before.numpy()))
print('Loss after: ', float(loss_after.numpy()))

## PyTorch: Weights And Activations With `torch.autograd`

PyTorch records the computation dynamically. Parameters collect gradients in `.grad` after `loss.backward()`. Intermediate activations need `retain_grad()` if you want to inspect their gradients after backpropagation.

In [ ]:
class TorchSmallConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_1 = nn.Conv2d(1, 8, kernel_size=3, padding=1, bias=False)
        self.relu_1 = nn.ReLU()
        self.pool_1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.dense_1 = nn.Linear(8 * 14 * 14, 16)
        self.relu_2 = nn.ReLU()
        self.logits = nn.Linear(16, 10)

    def forward(self, x):
        x = self.conv_1(x)
        x = self.relu_1(x)
        x = self.pool_1(x)
        x = self.flatten(x)
        x = self.dense_1(x)
        x = self.relu_2(x)
        return self.logits(x)


def choose_torch_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


torch_device = choose_torch_device()
torch_model = TorchSmallConvNet().to(torch_device)
torch_loss_fn = nn.CrossEntropyLoss()

x_torch_device = x_torch.to(torch_device)
y_torch_device = y_torch.to(torch_device)

activations = {}


def capture_activation(name):
    def hook(_module, _inputs, output):
        output.retain_grad()
        activations[name] = output
    return hook

hooks = [
    torch_model.conv_1.register_forward_hook(capture_activation('conv_1')),
    torch_model.relu_1.register_forward_hook(capture_activation('relu_1')),
    torch_model.dense_1.register_forward_hook(capture_activation('dense_1')),
]

torch_model.zero_grad(set_to_none=True)
logits = torch_model(x_torch_device)
loss = torch_loss_fn(logits, y_torch_device)
loss.backward()

print('Device:', torch_device)
print('Loss:', float(loss.detach().cpu()))
print()
print('Weight gradients')
for name, parameter in torch_model.named_parameters():
    print(f'{name:20} weight={tuple(parameter.shape)!s:16} grad={tuple(parameter.grad.shape)!s}')

print()
print('Activation gradients')
for name, activation in activations.items():
    print(f'{name:8} activation={tuple(activation.shape)!s:18} grad={tuple(activation.grad.shape)!s}')

## PyTorch: One Manual Training Step

This is the same learning loop in PyTorch form: zero old gradients, run forward, compute loss, backpropagate, then step the optimizer.

In [ ]:
torch_optimizer = torch.optim.SGD(torch_model.parameters(), lr=0.01)

torch_optimizer.zero_grad(set_to_none=True)
logits = torch_model(x_torch_device)
loss_before = torch_loss_fn(logits, y_torch_device)
loss_before.backward()
torch_optimizer.step()

with torch.no_grad():
    loss_after = torch_loss_fn(torch_model(x_torch_device), y_torch_device)

print('Loss before:', float(loss_before.detach().cpu()))
print('Loss after: ', float(loss_after.detach().cpu()))

for hook in hooks:
    hook.remove()

## Takeaways

The 2020 notebook used graph handles: model inputs, symbolic tensors, backend functions, and sessions.

* The modern TensorFlow version uses a tape around the forward pass. 
* The modern PyTorch version uses dynamic autograd and stores gradients on parameters after `backward()`.

The durable debugging habit is the same in all three versions: pick a scalar target, pick tensors of interest, compute gradients, and summarize them before staring at raw arrays.